# LeetCode 1559 — Detect Cycles in 2D Grid bằng Python

Link đề: https://leetcode.com/problems/detect-cycles-in-2d-grid/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Union-Find, Matrix

Skeleton LeetCode:

```python
class Solution:
    def containsCycle(self, grid: List[List[str]]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Xem lưới ký tự như một **đồ thị vô hướng**: mỗi ô là đỉnh, hai ô kề nhau cùng ký tự nối bằng cạnh; hiểu vì sao điều kiện "không quay lại ô vừa đi" và "độ dài ≥ 4" chính là **chu trình đơn** của đồ thị đó.
2. Phát hiện chu trình bằng **DFS/BFS nhớ ô cha**: gặp ô đã thăm mà không phải cha ⇒ có chu trình. Viết DFS **lặp** (stack) để tránh tràn đệ quy với lưới `500 × 500`.
3. Dùng **Union-Find (DSU)**: nối từng cạnh một lần; cạnh nối hai ô đã cùng gốc ⇒ chu trình.
4. Hiểu tiêu chí đồ thị học: một thành phần liên thông là **cây** ⇔ `E = V − 1`; có chu trình ⇔ `E ≥ V`.
5. Tránh các bẫy: không nhớ cha, so sánh nhầm ô khác ký tự, xử lý mỗi cạnh hai lần trong DSU, đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho lưới ký tự `grid` kích thước `m × n`. Hỏi có tồn tại **chu trình gồm các ô cùng giá trị** hay không.

- Một **chu trình** là đường đi có độ dài **≥ 4**, bắt đầu và kết thúc tại cùng một ô.
- Từ một ô, được đi sang một trong 4 ô kề (lên, xuống, trái, phải) **nếu ô đó cùng giá trị**.
- **Không được** đi ngược lại ô vừa đi ở bước trước. Ví dụ `(1,1) → (1,2) → (1,1)` là không hợp lệ.

Trả về `True` nếu có chu trình như vậy, ngược lại `False`.

**Ví dụ 1:** `grid = [["a","a","a","a"],["a","b","b","a"],["a","b","b","a"],["a","a","a","a"]]` → `True` (viền chữ `a` và khối `2×2` chữ `b` đều là chu trình).

**Ví dụ 2:** `grid = [["c","c","c","a"],["c","d","c","c"],["c","c","e","c"],["f","c","c","c"]]` → `True`.

**Ví dụ 3:** `grid = [["a","b","b"],["b","z","b"],["b","b","a"]]` → `False`.

**Ràng buộc:** `1 ≤ m, n ≤ 500`, `grid` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

def G(*rows):
    return [list(r) for r in rows]

examples = [
    (G("aaaa", "abba", "abba", "aaaa"), True),
    (G("ccca", "cdcc", "ccec", "fccc"), True),
    (G("abb", "bzb", "bba"), False),
]
for grid, want in examples:
    print("\n".join(" ".join(r) for r in grid), "-> mong đợi", want, "\n")

a a a a
a b b a
a b b a
a a a a -> mong đợi True 

c c c a
c d c c
c c e c
f c c c -> mong đợi True 

a b b
b z b
b b a -> mong đợi False 



## 3. Kiến thức nền

### 3.1 Duyệt 4 hướng trên lưới

Mảng hướng `DIRS = [(1,0), (-1,0), (0,1), (0,-1)]`; ô kề `(r+dr, c+dc)` hợp lệ khi nằm trong `[0, m) × [0, n)`.

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(grid, r, c):
    m, n = len(grid), len(grid[0])
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == grid[r][c]:
            yield nr, nc

g = examples[2][0]
print("hàng xóm cùng ký tự của (0,1):", list(neighbors(g, 0, 1)))
print("hàng xóm cùng ký tự của (1,1):", list(neighbors(g, 1, 1)))

hàng xóm cùng ký tự của (0,1): [(0, 2)]
hàng xóm cùng ký tự của (1,1): []


### 3.2 Đồ thị, cây và chu trình

Đồ thị vô hướng liên thông có `V` đỉnh: là **cây** (không chu trình) ⇔ có đúng `V − 1` cạnh. Nhiều cạnh hơn ⇒ chắc chắn có chu trình.

Lưới là đồ thị **hai phía** (tô màu bàn cờ: ô kề luôn khác màu), nên **không có chu trình độ dài 3**; và giữa hai ô chỉ có tối đa 1 cạnh, nên **không có chu trình độ dài 2** hợp lệ. Vậy mọi chu trình đơn trong đồ thị này tự động có độ dài ≥ 4 — điều kiện "≥ 4" của đề không cần kiểm tra riêng.

In [3]:
# khối 2x2 cùng chữ: V = 4, E = 4 >= V -> có chu trình (độ dài 4)
V, E = 4, 4
print("2x2:", "có chu trình" if E >= V else "cây")
# chữ L 3 ô: V = 3, E = 2 = V - 1 -> cây
print("L:", "có chu trình" if 2 >= 3 else "cây")

2x2: có chu trình
L: cây


### 3.3 DFS lặp bằng stack (tránh giới hạn đệ quy)

Python mặc định giới hạn đệ quy khoảng 1000 tầng; lưới `500 × 500` có thể tạo đường đi dài `250 000` ô. Dùng `list` làm stack, mỗi phần tử là `(ô, ô cha)`.

In [4]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())
stack = [((0, 0), None)]
cell, parent = stack.pop()
print("lấy ra:", cell, "cha:", parent)

giới hạn đệ quy mặc định: 1000
lấy ra: (0, 0) cha: None


### 3.4 Union-Find (Disjoint Set Union)

`find(x)` trả về gốc của tập chứa `x` (có **nén đường**), `union(a, b)` gộp hai tập. Nếu `find(a) == find(b)` trước khi gộp ⇒ `a`, `b` đã liên thông ⇒ thêm cạnh `a–b` sẽ tạo chu trình.

In [5]:
parent = list(range(5))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường (halving)
        x = parent[x]
    return x

for a, b in [(0, 1), (1, 2), (3, 4), (2, 0)]:
    ra, rb = find(a), find(b)
    print(f"cạnh {a}-{b}:", "TẠO CHU TRÌNH" if ra == rb else "gộp")
    parent[ra] = rb

cạnh 0-1: gộp
cạnh 1-2: gộp
cạnh 3-4: gộp
cạnh 2-0: TẠO CHU TRÌNH


### 3.5 Đánh số ô thành một số nguyên

Ô `(r, c)` ↔ số `r * n + c`. Tiện cho mảng `parent` của DSU và mảng `visited` một chiều.

In [6]:
m, n = 3, 4
for r, c in [(0, 0), (1, 2), (2, 3)]:
    idx = r * n + c
    print((r, c), "->", idx, "->", divmod(idx, n))

(0, 0) -> 0 -> (0, 0)
(1, 2) -> 6 -> (1, 2)
(2, 3) -> 11 -> (2, 3)


## 4. Ý tưởng nền tảng / trực giác

Xét khối `2 × 2` cùng chữ `b` trong ví dụ 1 (các ô `(1,1), (1,2), (2,1), (2,2)`). Đi DFS và nhớ "mình đến từ đâu":

1. Bắt đầu `(1,1)`, cha = không có.
2. Sang `(2,1)` (cha `(1,1)`).
3. Từ `(2,1)` sang `(2,2)` (cha `(2,1)`).
4. Từ `(2,2)` nhìn lên `(1,2)` — chưa thăm, đi tiếp (cha `(2,2)`).
5. Từ `(1,2)` nhìn sang trái `(1,1)` — **đã thăm** và **không phải cha** `(2,2)` ⇒ có một đường khác quay về ⇒ **chu trình** `(1,1)→(2,1)→(2,2)→(1,2)→(1,1)` dài 4.

Ngược lại, ở ví dụ 3 các ô chữ `b` tạo thành **hai** nhóm hình chữ L không khép kín (hai chữ `a` ở góc chặn lại): `{(0,1),(0,2),(1,2)}` và `{(1,0),(2,0),(2,1)}`. DFS trong mỗi nhóm chỉ gặp lại ô đã thăm là **ô cha** (đường vừa đi) ⇒ không có chu trình.

Cách nhìn thứ hai — **đếm**: mỗi nhóm `b` có `V = 3` ô và `E = 2` cạnh `= V − 1` ⇒ là cây; nhóm khối `2×2` ở ví dụ 1 có `V = 4`, `E = 4 ≥ V` ⇒ có chu trình. Ở ví dụ 3, mọi nhóm đều là cây ⇒ `False`.

In [7]:
def components_stats(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    out = []
    for r in range(m):
        for c in range(n):
            if seen[r][c]:
                continue
            seen[r][c] = True
            q, cells, deg = [(r, c)], 0, 0
            while q:
                x, y = q.pop()
                cells += 1
                for nx, ny in neighbors(grid, x, y):
                    deg += 1
                    if not seen[nx][ny]:
                        seen[nx][ny] = True
                        q.append((nx, ny))
            out.append((grid[r][c], cells, deg // 2))
    return out

for grid, want in examples:
    print([(ch, f"V={v}", f"E={e}", "CHU TRÌNH" if e >= v else "cây") for ch, v, e in components_stats(grid)])

[('a', 'V=12', 'E=12', 'CHU TRÌNH'), ('b', 'V=4', 'E=4', 'CHU TRÌNH')]
[('c', 'V=12', 'E=12', 'CHU TRÌNH'), ('a', 'V=1', 'E=0', 'cây'), ('d', 'V=1', 'E=0', 'cây'), ('e', 'V=1', 'E=0', 'cây'), ('f', 'V=1', 'E=0', 'cây')]
[('a', 'V=1', 'E=0', 'cây'), ('b', 'V=3', 'E=2', 'cây'), ('b', 'V=3', 'E=2', 'cây'), ('z', 'V=1', 'E=0', 'cây'), ('a', 'V=1', 'E=0', 'cây')]


## 5. Cách 1 — Brute Force / baseline: tìm đường đi khép kín bằng quay lui

**Ý tưởng.** Làm đúng định nghĩa: với mỗi ô xuất phát `s`, quay lui liệt kê mọi đường đi **đơn** (không lặp ô) qua các ô cùng chữ; nếu từ ô hiện tại có thể bước về `s` và đường đã dài ≥ 3 cạnh (tức chu trình ≥ 4 bước) thì trả `True`.

Vì đường đi đơn không lặp ô, quy tắc "không quay lại ô vừa đi" tự động thoả. Đây là chuẩn đối chiếu **trực tiếp theo đề**, nhưng số đường đi đơn tăng theo hàm mũ ⇒ chỉ dùng cho lưới rất nhỏ.

In [8]:
def brute_backtrack(grid):
    m, n = len(grid), len(grid[0])

    def dfs(s, cur, length, on_path):
        for nb in neighbors(grid, *cur):
            if nb == s and length >= 3:          # quay về điểm đầu sau >= 3 cạnh -> chu trình >= 4
                return True
            if nb not in on_path:
                on_path.add(nb)
                if dfs(s, nb, length + 1, on_path):
                    return True
                on_path.remove(nb)
        return False

    for r in range(m):
        for c in range(n):
            if dfs((r, c), (r, c), 0, {(r, c)}):
                return True
    return False

for grid, want in examples:
    print(brute_backtrack(grid), "mong đợi", want)

True mong đợi True
True mong đợi True
False mong đợi False


In [9]:
# Nút thắt: ô (0,0) là "cuống" 'a' treo vào một khối k x k chữ 'a'.
# Quay lui bắt đầu từ (0,0) phải liệt kê MỌI đường đi đơn trong khối mà không bao giờ quay về được (0,0).
def pendant_grid(k):
    return [list("a" + "b" * (k - 1))] + [list("a" * k) for _ in range(k)]

for k in [3, 4, 5]:
    g = pendant_grid(k)
    t0 = time.perf_counter()
    r = brute_backtrack(g)
    print(f"khối {k}x{k}: {r} ({time.perf_counter() - t0:.4f}s)")

# Dùng thêm cho các phần sau: lưới "răng lược" lớn không có chu trình
def comb_grid(m, n):
    # hàng 0 toàn 'a', các cột chẵn kéo 'a' xuống; cột lẻ (từ hàng 1) là đoạn thẳng 'b'
    return [["a" if r == 0 or c % 2 == 0 else "b" for c in range(n)] for r in range(m)]

print("răng lược 4x7 (không chu trình):", brute_backtrack(comb_grid(4, 7)))

khối 3x3: True (0.0001s)
khối 4x4: True (0.0022s)


khối 5x5: True (0.1663s)
răng lược 4x7 (không chu trình): False


**Độ phức tạp:** mũ theo số ô (số đường đi đơn). **Nút thắt:** ta liệt kê lại các đường đi chung tiền tố rất nhiều lần. Thực ra chỉ cần **một lần duyệt** mỗi thành phần: một khi gặp lại ô đã thăm qua một cạnh khác cạnh cha là đủ kết luận.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chuyển sang ngôn ngữ đồ thị.** Đồ thị `Gc`: đỉnh = ô, cạnh = cặp ô kề cùng chữ. Chu trình theo đề ⇔ chu trình đơn trong `Gc` (độ dài ≥ 4 tự động do lưới hai phía, xem 3.2).

**Insight 2 — DFS nhớ cha.** Trong DFS trên đồ thị vô hướng, *nếu* từ ô `u` thấy ô kề `v` **đã thăm** và `v` **không phải cha** của `u` *thì* có chu trình (cạnh `u–v` là "cạnh ngược", cùng với đường trên cây DFS tạo thành vòng). *Nếu* duyệt xong mà không gặp ⇒ thành phần là cây.

**Insight 3 — DSU.** Duyệt mỗi cạnh **đúng một lần** (chỉ nhìn sang phải và xuống dưới). *Nếu* hai đầu cạnh đã cùng gốc *thì* cạnh này đóng một chu trình. Ngược lại gộp hai tập.

**Insight 4 — đếm.** Thành phần có `E ≥ V` ⇔ có chu trình. Đếm `V`, `E` mỗi thành phần cũng là một lời giải `O(mn)`.

**Lưu ý với BFS:** BFS cũng nhớ cha và áp dụng cùng quy tắc; với đồ thị vô hướng, quy tắc "gặp ô đã thăm không phải cha" vẫn đúng cho BFS.

In [10]:
# Minh hoạ: cạnh nào DSU phát hiện đóng chu trình đầu tiên trong ví dụ 1
grid = examples[0][0]
m, n = len(grid), len(grid[0])
par = list(range(m * n))
def f(x):
    while par[x] != x:
        par[x] = par[par[x]]
        x = par[x]
    return x
done = False
for r in range(m):
    for c in range(n):
        for dr, dc in ((0, 1), (1, 0)):
            nr, nc = r + dr, c + dc
            if nr < m and nc < n and grid[nr][nc] == grid[r][c] and not done:
                a, b = f(r * n + c), f(nr * n + nc)
                if a == b:
                    print(f"cạnh ({r},{c})-({nr},{nc}) chữ {grid[r][c]!r} nối hai ô đã cùng tập -> chu trình")
                    done = True
                par[a] = b

cạnh (2,1)-(2,2) chữ 'b' nối hai ô đã cùng tập -> chu trình


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm `V` và `E` của mỗi thành phần

Từng đoạn:
1. BFS/DFS lặp từ mỗi ô chưa thăm, đếm số ô `V` và tổng bậc (mỗi cạnh được đếm 2 lần) ⇒ `E = tổng bậc / 2`.
2. Nếu `E ≥ V` ⇒ `True`.

Ưu điểm: không cần lý luận về "cha"; chỉ là đếm.

In [11]:
def count_v_e(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    for r in range(m):
        for c in range(n):
            if seen[r][c]:
                continue
            seen[r][c] = True
            stack, V, deg = [(r, c)], 0, 0
            ch = grid[r][c]
            while stack:
                x, y = stack.pop()
                V += 1
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == ch:
                        deg += 1
                        if not seen[nx][ny]:
                            seen[nx][ny] = True
                            stack.append((nx, ny))
            if deg // 2 >= V:
                return True
    return False

print([count_v_e(g) for g, _ in examples])

[True, True, False]


### 7.2 Cách 3 — DFS lặp nhớ ô cha

Từng đoạn:
1. `visited` toàn `False`. Với mỗi ô chưa thăm, đẩy `(ô, cha = None)` vào stack, đánh dấu thăm.
2. Lấy `(u, p)` ra; xét mọi ô kề `v` cùng chữ:
   - `v == p` ⇒ bỏ qua (đây là cạnh vừa đi).
   - `v` đã thăm ⇒ **chu trình** ⇒ `True`.
   - Ngược lại đánh dấu thăm, đẩy `(v, u)`.

**Chú ý:** đánh dấu thăm **khi đẩy vào stack** (không phải khi lấy ra) để mỗi ô vào stack đúng một lần; khi đó "gặp ô đã thăm không phải cha" vẫn tương ứng với một cạnh ngoài cây ⇒ chu trình. (Lập luận chi tiết ở mục 10.)

In [12]:
def dfs_parent(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    for sr in range(m):
        for sc in range(n):
            if visited[sr][sc]:
                continue
            ch = grid[sr][sc]
            visited[sr][sc] = True
            stack = [(sr, sc, -1, -1)]
            while stack:
                r, c, pr, pc = stack.pop()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n) or grid[nr][nc] != ch:
                        continue
                    if nr == pr and nc == pc:
                        continue
                    if visited[nr][nc]:
                        return True
                    visited[nr][nc] = True
                    stack.append((nr, nc, r, c))
    return False

for grid, want in examples:
    print(dfs_parent(grid), "mong đợi", want)

True mong đợi True
True mong đợi True
False mong đợi False


Code nộp LeetCode cho cách 3:

In [13]:
class SolutionDFS:
    def containsCycle(self, grid: List[List[str]]) -> bool:
        m, n = len(grid), len(grid[0])
        visited = [[False] * n for _ in range(m)]
        for sr in range(m):
            for sc in range(n):
                if visited[sr][sc]:
                    continue
                ch = grid[sr][sc]
                visited[sr][sc] = True
                stack = [(sr, sc, -1, -1)]
                while stack:
                    r, c, pr, pc = stack.pop()
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == ch and (nr, nc) != (pr, pc):
                            if visited[nr][nc]:
                                return True
                            visited[nr][nc] = True
                            stack.append((nr, nc, r, c))
        return False

print([SolutionDFS().containsCycle(g) for g, _ in examples])

[True, True, False]


**Độ phức tạp:** mỗi ô vào stack 1 lần, xét 4 hàng xóm ⇒ Time `O(m·n)`, Space `O(m·n)`.

### 7.3 Cách 4 — BFS nhớ ô cha

Giống hệt cách 3 nhưng dùng `deque` (FIFO). Kết quả giống nhau; BFS đôi khi phát hiện chu trình ngắn sớm hơn.

In [14]:
class SolutionBFS:
    def containsCycle(self, grid: List[List[str]]) -> bool:
        m, n = len(grid), len(grid[0])
        visited = [[False] * n for _ in range(m)]
        for sr in range(m):
            for sc in range(n):
                if visited[sr][sc]:
                    continue
                ch = grid[sr][sc]
                visited[sr][sc] = True
                q = deque([(sr, sc, -1, -1)])
                while q:
                    r, c, pr, pc = q.popleft()
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == ch and (nr, nc) != (pr, pc):
                            if visited[nr][nc]:
                                return True
                            visited[nr][nc] = True
                            q.append((nr, nc, r, c))
        return False

print([SolutionBFS().containsCycle(g) for g, _ in examples])

[True, True, False]


### 7.4 Cách 5 — Union-Find

Từng đoạn:
1. `parent[i] = i` cho mọi ô `i = r*n + c`.
2. Với mỗi ô, chỉ xét cạnh sang **phải** và **xuống** (mỗi cạnh đúng 1 lần) nếu cùng chữ.
3. `find` hai đầu; cùng gốc ⇒ `True`; khác ⇒ gộp (union by size để cây nông).

In [15]:
class SolutionDSU:
    def containsCycle(self, grid: List[List[str]]) -> bool:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n))
        size = [1] * (m * n)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for r in range(m):
            row = grid[r]
            for c in range(n):
                ch = row[c]
                for nr, nc in ((r, c + 1), (r + 1, c)):
                    if nr < m and nc < n and grid[nr][nc] == ch:
                        a, b = find(r * n + c), find(nr * n + nc)
                        if a == b:
                            return True
                        if size[a] < size[b]:
                            a, b = b, a
                        parent[b] = a
                        size[a] += size[b]
        return False

print([SolutionDSU().containsCycle(g) for g, _ in examples])

[True, True, False]


**Độ phức tạp:** `O(m·n · α(m·n))` ≈ `O(m·n)`, Space `O(m·n)`. Không cần stack/đệ quy.

## 8. TRACE từng bước

Trace DFS lặp trên lưới nhỏ `["aa","aa"]` (khối `2×2`):

| Bước | Lấy ra (ô, cha) | Xét hàng xóm | Hành động |
|------|-----------------|--------------|-----------|
| 1 | (0,0), – | (1,0) mới; (0,1) mới | đẩy (1,0) và (0,1) |
| 2 | (0,1), (0,0) | (1,1) mới; (0,0) là cha | đẩy (1,1) |
| 3 | (1,1), (0,1) | (1,0) **đã thăm**, không phải cha | **chu trình** ⇒ True |

In [16]:
def trace_dfs(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    step = 0
    for sr in range(m):
        for sc in range(n):
            if visited[sr][sc]:
                continue
            ch = grid[sr][sc]
            visited[sr][sc] = True
            stack = [((sr, sc), None)]
            print(f"bắt đầu thành phần chữ {ch!r} tại {(sr, sc)}")
            while stack:
                (r, c), p = stack.pop()
                step += 1
                notes = []
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n) or grid[nr][nc] != ch:
                        continue
                    if (nr, nc) == p:
                        notes.append(f"{(nr, nc)} là cha")
                        continue
                    if visited[nr][nc]:
                        notes.append(f"{(nr, nc)} ĐÃ THĂM, không phải cha")
                        print(f"  bước {step}: lấy {(r, c)} (cha {p}) | " + "; ".join(notes) + " -> True")
                        return True
                    visited[nr][nc] = True
                    stack.append(((nr, nc), (r, c)))
                    notes.append(f"{(nr, nc)} mới -> đẩy")
                print(f"  bước {step}: lấy {(r, c)} (cha {p}) | " + ("; ".join(notes) if notes else "không có hàng xóm cùng chữ"))
    print("không có chu trình -> False")
    return False

trace_dfs(G("aa", "aa"))

bắt đầu thành phần chữ 'a' tại (0, 0)
  bước 1: lấy (0, 0) (cha None) | (1, 0) mới -> đẩy; (0, 1) mới -> đẩy
  bước 2: lấy (0, 1) (cha (0, 0)) | (1, 1) mới -> đẩy; (0, 0) là cha
  bước 3: lấy (1, 1) (cha (0, 1)) | (0, 1) là cha; (1, 0) ĐÃ THĂM, không phải cha -> True


True

In [17]:
trace_dfs(G("abb", "bzb", "bba"))

bắt đầu thành phần chữ 'a' tại (0, 0)
  bước 1: lấy (0, 0) (cha None) | không có hàng xóm cùng chữ
bắt đầu thành phần chữ 'b' tại (0, 1)
  bước 2: lấy (0, 1) (cha None) | (0, 2) mới -> đẩy
  bước 3: lấy (0, 2) (cha (0, 1)) | (1, 2) mới -> đẩy; (0, 1) là cha
  bước 4: lấy (1, 2) (cha (0, 2)) | (0, 2) là cha
bắt đầu thành phần chữ 'b' tại (1, 0)
  bước 5: lấy (1, 0) (cha None) | (2, 0) mới -> đẩy
  bước 6: lấy (2, 0) (cha (1, 0)) | (1, 0) là cha; (2, 1) mới -> đẩy
  bước 7: lấy (2, 1) (cha (2, 0)) | (2, 0) là cha
bắt đầu thành phần chữ 'z' tại (1, 1)
  bước 8: lấy (1, 1) (cha None) | không có hàng xóm cùng chữ
bắt đầu thành phần chữ 'a' tại (2, 2)
  bước 9: lấy (2, 2) (cha None) | không có hàng xóm cùng chữ
không có chu trình -> False


False

Trace Union-Find trên ví dụ 2 — in từng cạnh (phải/xuống) được xét và trạng thái gộp:

In [18]:
def trace_dsu(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for r in range(m):
        for c in range(n):
            for nr, nc in ((r, c + 1), (r + 1, c)):
                if nr < m and nc < n and grid[nr][nc] == grid[r][c]:
                    a, b = find(r * n + c), find(nr * n + nc)
                    if a == b:
                        print(f"  cạnh ({r},{c})-({nr},{nc}) chữ {grid[r][c]!r}: cùng gốc {a} -> CHU TRÌNH")
                        return True
                    parent[b] = a
                    print(f"  cạnh ({r},{c})-({nr},{nc}) chữ {grid[r][c]!r}: gộp gốc {b} vào {a}")
    return False

trace_dsu(examples[1][0])

  cạnh (0,0)-(0,1) chữ 'c': gộp gốc 1 vào 0
  cạnh (0,0)-(1,0) chữ 'c': gộp gốc 4 vào 0
  cạnh (0,1)-(0,2) chữ 'c': gộp gốc 2 vào 0
  cạnh (0,2)-(1,2) chữ 'c': gộp gốc 6 vào 0
  cạnh (1,0)-(2,0) chữ 'c': gộp gốc 8 vào 0
  cạnh (1,2)-(1,3) chữ 'c': gộp gốc 7 vào 0
  cạnh (1,3)-(2,3) chữ 'c': gộp gốc 11 vào 0
  cạnh (2,0)-(2,1) chữ 'c': gộp gốc 9 vào 0
  cạnh (2,1)-(3,1) chữ 'c': gộp gốc 13 vào 0
  cạnh (2,3)-(3,3) chữ 'c': gộp gốc 15 vào 0
  cạnh (3,1)-(3,2) chữ 'c': gộp gốc 14 vào 0
  cạnh (3,2)-(3,3) chữ 'c': cùng gốc 0 -> CHU TRÌNH


True

## 9. Code nộp LeetCode

Bản nộp chính: DFS lặp nhớ cha (không lo giới hạn đệ quy). `SolutionDSU` ở mục 7.4 cũng là một bản nộp tốt.

In [19]:
class Solution:
    def containsCycle(self, grid: List[List[str]]) -> bool:
        m, n = len(grid), len(grid[0])
        visited = [[False] * n for _ in range(m)]
        for sr in range(m):
            for sc in range(n):
                if visited[sr][sc]:
                    continue
                ch = grid[sr][sc]
                visited[sr][sc] = True
                stack = [(sr, sc, -1, -1)]               # (hàng, cột, hàng cha, cột cha)
                while stack:
                    r, c, pr, pc = stack.pop()
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if not (0 <= nr < m and 0 <= nc < n) or grid[nr][nc] != ch:
                            continue
                        if nr == pr and nc == pc:        # không đi ngược cạnh vừa đi
                            continue
                        if visited[nr][nc]:              # gặp lại ô đã thăm qua cạnh khác -> chu trình
                            return True
                        visited[nr][nc] = True
                        stack.append((nr, nc, r, c))
        return False

for grid, want in examples:
    got = Solution().containsCycle(grid)
    print(got, want)
    assert got == want

True True
True True
False False


## 10. Vì sao thuật toán đúng

**(a) Chu trình theo đề ⇔ chu trình đơn trong đồ thị `Gc`.** Một chu trình đơn trong `Gc` có độ dài ≥ 3; lưới hai phía nên độ dài chẵn ⇒ ≥ 4, và không bao giờ bước ngược cạnh vừa đi ⇒ hợp lệ theo đề. Ngược lại, một đường khép kín hợp lệ theo đề (không quay đầu tức thì) trong đồ thị vô hướng luôn chứa một chu trình đơn (lấy đoạn giữa hai lần xuất hiện gần nhất của một ô; đoạn đó không thể dài 2 vì không quay đầu).

**(b) DFS/BFS nhớ cha.** Mỗi ô được đánh dấu đúng một lần, ngay khi được đẩy vào stack từ một ô cha ⇒ các cạnh `(x, cha(x))` tạo thành một **rừng khung** của `Gc`. Khi lấy `u` ra và thấy hàng xóm `v` (cùng chữ) đã đánh dấu, `v ≠ cha(u)`: cạnh `u–v` chỉ có thể là cạnh cây nếu `v = cha(u)` (đã loại) hoặc `u = cha(v)`. Nhưng `u` chỉ được lấy ra **một lần**, và `v` chỉ trở thành con của `u` nếu được đẩy **trong lần xét này, sau phép kiểm tra** — mà `v` đã được đánh dấu từ trước. Vậy `u–v` là cạnh **ngoài** rừng khung; thêm một cạnh ngoài vào cây khung luôn tạo chu trình ⇒ trả `True` là đúng.

Ngược lại nếu có chu trình, thành phần chứa nó có `E ≥ V` nên có ít nhất một cạnh ngoài cây khung; cạnh đó được xét từ một trong hai đầu (đầu được lấy ra sau), khi đó đầu kia đã thăm và không là cha ⇒ thuật toán trả `True`.

**(c) DSU.** Duyệt mỗi cạnh một lần; cạnh nối hai ô đã cùng tập là cạnh ngoài cây khung ⇒ chu trình; nếu không có cạnh nào như vậy thì tập cạnh là một rừng ⇒ không chu trình.

In [20]:
# thực nghiệm (a): brute force theo định nghĩa == tiêu chí E >= V trên lưới nhỏ ngẫu nhiên
rng = random.Random(10)
for _ in range(400):
    m, n = rng.randint(1, 4), rng.randint(1, 4)
    g = [[rng.choice("ab") for _ in range(n)] for _ in range(m)]
    assert brute_backtrack(g) == count_v_e(g) == dfs_parent(g)
print("định nghĩa đề == E >= V == DFS nhớ cha: OK")

định nghĩa đề == E >= V == DFS nhớ cha: OK


## 11. Phân tích Time / Space

Gọi `N = m·n` (≤ `250 000`).

| Cách | Time | Space |
|------|------|-------|
| Quay lui đường đi đơn | mũ | `O(N)` |
| Đếm V, E | `O(N)` | `O(N)` |
| DFS lặp nhớ cha | `O(N)` | `O(N)` (visited + stack) |
| BFS nhớ cha | `O(N)` | `O(N)` |
| Union-Find | `O(N · α(N))` | `O(N)` |

Trường hợp xấu nhất về thời gian là khi **không** có chu trình (phải duyệt hết); ví dụ lưới "răng lược" lớn.

In [21]:
big_nocycle = comb_grid(500, 500)
rng = random.Random(1559)
big_random = [[rng.choice("ab") for _ in range(500)] for _ in range(500)]
for name, S in [("DFS", SolutionDFS), ("BFS", SolutionBFS), ("DSU", SolutionDSU), ("Solution", Solution)]:
    t0 = time.perf_counter()
    r1 = S().containsCycle(big_nocycle)
    t1 = time.perf_counter()
    r2 = S().containsCycle(big_random)
    t2 = time.perf_counter()
    print(f"{name:8s}: răng lược 500x500 -> {r1} ({t1 - t0:.3f}s) | ngẫu nhiên -> {r2} ({t2 - t1:.4f}s)")

DFS     : răng lược 500x500 -> False (0.203s) | ngẫu nhiên -> True (0.0010s)


BFS     : răng lược 500x500 -> False (0.228s) | ngẫu nhiên -> True (0.0010s)
DSU     : răng lược 500x500 -> False (0.150s) | ngẫu nhiên -> True (0.0086s)


Solution: răng lược 500x500 -> False (0.214s) | ngẫu nhiên -> True (0.0003s)


## 12. Bộ test

In [22]:
METHODS = {
    "brute_backtrack": brute_backtrack,
    "count_v_e": count_v_e,
    "dfs_parent": dfs_parent,
    "SolutionDFS": lambda g: SolutionDFS().containsCycle(g),
    "SolutionBFS": lambda g: SolutionBFS().containsCycle(g),
    "SolutionDSU": lambda g: SolutionDSU().containsCycle(g),
    "Solution": lambda g: Solution().containsCycle(g),
}
for grid, want in examples:
    for name, f in METHODS.items():
        assert f(grid) == want, name
print("ví dụ đề: OK")

ví dụ đề: OK


In [23]:
# edge case
edge = [
    (G("a"), False),                     # 1 ô
    (G("aa"), False),                    # 2 ô: đi qua đi lại không tính
    (G("aaaa"), False),                  # một hàng
    (G("a", "a", "a"), False),           # một cột
    (G("aa", "aa"), True),               # khối 2x2 nhỏ nhất
    (G("ab", "ba"), False),              # bàn cờ
    (G("aaa", "aba", "aaa"), True),      # vòng quanh tâm khác chữ
    (G("aaa", "aba", "abb"), False),     # chữ 'a' hình chữ U
    (G("abab", "baba"), False),
]
for grid, want in edge:
    for name, f in METHODS.items():
        assert f(grid) == want, (name, grid)
print("edge case: OK")

edge case: OK


In [24]:
# ngẫu nhiên có seed: đối chiếu với quay lui theo định nghĩa
rng = random.Random(2022)
cnt_true = 0
for _ in range(1500):
    m, n = rng.randint(1, 4), rng.randint(1, 5)
    alpha = rng.choice(["ab", "abc", "aab"])
    g = [[rng.choice(alpha) for _ in range(n)] for _ in range(m)]
    ref = brute_backtrack(g)
    cnt_true += ref
    for name, f in METHODS.items():
        assert f(g) == ref, (name, g)
print(f"1500 lưới ngẫu nhiên (seed 2022): OK, {cnt_true} lưới có chu trình")

1500 lưới ngẫu nhiên (seed 2022): OK, 297 lưới có chu trình


In [25]:
# lưới lớn hơn: các cách O(N) khớp nhau
rng = random.Random(8)
for _ in range(40):
    m, n = rng.randint(1, 60), rng.randint(1, 60)
    alpha = rng.choice(["ab", "abc", "abcdefgh"])
    g = [[rng.choice(alpha) for _ in range(n)] for _ in range(m)]
    ref = count_v_e(g)
    for name in ["dfs_parent", "SolutionDFS", "SolutionBFS", "SolutionDSU", "Solution"]:
        assert METHODS[name](g) == ref
print("40 lưới cỡ vừa: OK")

40 lưới cỡ vừa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không nhớ ô cha

Coi mọi hàng xóm đã thăm là chu trình ⇒ cạnh vừa đi (quay về cha) bị tính là chu trình.

In [26]:
def wrong_no_parent(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    for sr in range(m):
        for sc in range(n):
            if visited[sr][sc]:
                continue
            visited[sr][sc] = True
            stack = [(sr, sc)]
            while stack:
                r, c = stack.pop()
                for nr, nc in neighbors(grid, r, c):
                    if visited[nr][nc]:
                        return True              # SAI: có thể chỉ là ô cha
                    visited[nr][nc] = True
                    stack.append((nr, nc))
    return False

for grid in [G("aa"), G("aaa"), examples[2][0]]:
    print(grid, "| sai:", wrong_no_parent(grid), "| đúng:", Solution().containsCycle(grid))

[['a', 'a']] | sai: True | đúng: False
[['a', 'a', 'a']] | sai: True | đúng: False
[['a', 'b', 'b'], ['b', 'z', 'b'], ['b', 'b', 'a']] | sai: True | đúng: False


Sửa: lưu cha cùng mỗi ô trong stack và bỏ qua hàng xóm là cha.

### Lỗi 2 — Quên kiểm tra cùng ký tự khi xét "đã thăm"

`visited` dùng chung cho cả lưới. Nếu không lọc `grid[nr][nc] == ch`, ô đã thăm của **chữ khác** bị coi là chu trình.

In [27]:
def wrong_any_char(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    for sr in range(m):
        for sc in range(n):
            if visited[sr][sc]:
                continue
            ch = grid[sr][sc]
            visited[sr][sc] = True
            stack = [(sr, sc, -1, -1)]
            while stack:
                r, c, pr, pc = stack.pop()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n) or (nr, nc) == (pr, pc):
                        continue
                    if visited[nr][nc]:
                        return True              # SAI: chưa kiểm tra grid[nr][nc] == ch
                    if grid[nr][nc] == ch:
                        visited[nr][nc] = True
                        stack.append((nr, nc, r, c))
    return False

for grid in [examples[2][0], G("ab", "ba"), G("aab")]:
    print(grid, "| sai:", wrong_any_char(grid), "| đúng:", Solution().containsCycle(grid))

[['a', 'b', 'b'], ['b', 'z', 'b'], ['b', 'b', 'a']] | sai: True | đúng: False
[['a', 'b'], ['b', 'a']] | sai: True | đúng: False
[['a', 'a', 'b']] | sai: True | đúng: False


Sửa: chỉ xét hàng xóm **cùng ký tự** rồi mới kiểm tra `visited`.

### Lỗi 3 — Union-Find xét cả 4 hướng (mỗi cạnh bị xử lý 2 lần)

Cạnh `u–v` được gộp khi xét `u`, rồi khi xét `v` lại thấy `u`, `v` cùng gốc ⇒ báo chu trình sai.

In [28]:
def wrong_dsu_four_dirs(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for r in range(m):
        for c in range(n):
            for nr, nc in neighbors(grid, r, c):     # SAI: cả 4 hướng
                a, b = find(r * n + c), find(nr * n + nc)
                if a == b:
                    return True
                parent[a] = b
    return False

for grid in [G("aa"), G("abc"), examples[2][0]]:
    print(grid, "| sai:", wrong_dsu_four_dirs(grid), "| đúng:", Solution().containsCycle(grid))

[['a', 'a']] | sai: True | đúng: False
[['a', 'b', 'c']] | sai: False | đúng: False
[['a', 'b', 'b'], ['b', 'z', 'b'], ['b', 'b', 'a']] | sai: True | đúng: False


Sửa: chỉ xét hướng **phải** và **xuống**.

### Lỗi 4 — DFS đệ quy với giới hạn mặc định

Đúng về logic nhưng với đường đi dài (lưới "con rắn") độ sâu đệ quy vượt ~1000 ⇒ `RecursionError` (trên LeetCode là Runtime Error). Ta bắt lỗi để minh hoạ.

In [29]:
def recursive_dfs(grid):
    m, n = len(grid), len(grid[0])
    visited = [[False] * n for _ in range(m)]
    def dfs(r, c, pr, pc):
        visited[r][c] = True
        for nr, nc in neighbors(grid, r, c):
            if (nr, nc) == (pr, pc):
                continue
            if visited[nr][nc] or dfs(nr, nc, r, c):
                return True
        return False
    return any(not visited[r][c] and dfs(r, c, -1, -1) for r in range(m) for c in range(n))

def snake_grid(m, n):
    # hàng chẵn toàn 'a'; hàng lẻ toàn 'b' trừ một ô 'a' nối ở đầu/cuối luân phiên -> con rắn 'a' dài ~ m*n/2, không chu trình
    g = [["a"] * n if r % 2 == 0 else ["b"] * n for r in range(m)]
    for r in range(1, m, 2):
        g[r][n - 1 if (r // 2) % 2 == 0 else 0] = "a"
    return g

for size in [(5, 9), (60, 60)]:
    g = snake_grid(*size)
    try:
        print(size, "đệ quy:", recursive_dfs(g))
    except RecursionError as e:
        print(size, "đệ quy: RecursionError -", str(e)[:50], "| DFS lặp:", Solution().containsCycle(g))

(5, 9) đệ quy: False
(60, 60) đệ quy: RecursionError - maximum recursion depth exceeded | DFS lặp: False


Sửa: dùng DFS **lặp** (stack), BFS hoặc DSU. (Tăng `sys.setrecursionlimit` cũng có thể gây tràn stack C với lưới 500×500.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lưới + đi 4 hướng + cùng giá trị" ⇒ đồ thị ẩn; "chu trình" trong đồ thị **vô hướng** ⇒ DFS/BFS **nhớ cha** hoặc **Union-Find**.
- Với DSU trên lưới: chỉ nối **phải** và **xuống** để mỗi cạnh được xét một lần.
- Lưới lớn (`> 10⁴` ô) ⇒ tránh đệ quy trong Python.
- Điều kiện "độ dài ≥ 4" trên lưới thường là "miễn phí" (lưới hai phía, không có tam giác).
- Mẹo kiểm chứng nhanh: `E ≥ V` trong một thành phần ⇔ có chu trình.

## 15. Biến thể của bài

1. **Đồ thị có hướng**: phải dùng 3 màu (trắng/xám/đen) — Course Schedule (LC 207).
2. **Redundant Connection (LC 684)**: tìm cạnh đóng chu trình đầu tiên — DSU y hệt cách 5.
3. **Đếm số chu trình độc lập** (số vòng cơ sở): `Σ (E − V + 1)` qua các thành phần — bài tập 5 biến thể.
4. **Chu trình cho phép đi chéo (8 hướng)**: khi đó có tam giác; điều kiện "độ dài ≥ 4" trở nên quan trọng.
5. **Number of Closed Islands (LC 1254)**, **Surrounded Regions (LC 130)**: cùng khuôn duyệt lưới.
6. Trả về **các chữ cái** có chu trình (bài tập 5).

In [30]:
# Biến thể 3: số chu trình độc lập (cyclomatic number) = Σ (E - V + 1) qua mọi thành phần
def cyclomatic(grid):
    return sum(e - v + 1 for _, v, e in components_stats(grid))

print([cyclomatic(g) for g, _ in examples], cyclomatic(G("aaa", "aaa", "aaa")))

[2, 1, 0] 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Quay lui | liệt kê đường đi đơn khép kín | mũ | `O(N)` | kiểm chứng, lưới ≤ 20 ô |
| 2. Đếm V, E | thành phần có `E ≥ V` | `O(N)` | `O(N)` | lý luận đơn giản, cần thống kê thành phần |
| 3. DFS lặp nhớ cha | cạnh ngược ⇒ chu trình | `O(N)` | `O(N)` | **bản nộp** |
| 4. BFS nhớ cha | như 3 với hàng đợi | `O(N)` | `O(N)` | quen BFS |
| 5. Union-Find | cạnh nối hai ô cùng gốc | `O(N α(N))` | `O(N)` | không cần stack, dễ mở rộng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `containsCycle` của `[["a","a"],["a","a"]]` và `[["a","b"],["b","a"]]`?

**Bài 2 (tính tay):** Với lưới `["aab","aab","bbb"]`, liệt kê các thành phần, `V`, `E` của từng thành phần và kết luận.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_find(parent, x)` — tìm gốc có nén đường.

**Bài 4 (tự viết lại):** Viết `ex4_bfs(grid)` phát hiện chu trình bằng BFS nhớ cha (không xem lại code mục 7.3).

**Bài 5 (mini challenge):** Viết `ex5_cycle_letters(grid)` trả về **tập các chữ cái** có ít nhất một chu trình.

In [31]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_find(parent, x):
    while parent[x] != x:
        # TODO: nén đường: trỏ parent[x] lên ông của x
        # TODO: nhảy x lên cha
        pass
    return x

In [32]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_bfs(grid):
    # TODO
    raise NotImplementedError

def ex5_cycle_letters(grid):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Khối 2×2 cùng chữ là chu trình nhỏ nhất. (2) Bàn cờ: mỗi ô cô lập. (3) `True`, `False`.

**Bài 2.** (1) Chữ `a`: khối 2×2 ở góc trái trên. (2) Chữ `b`: 5 ô hình chữ L ngược (cột phải + hàng dưới). (3) `a`: `V = 4, E = 4` ⇒ chu trình; `b`: `V = 5, E = 4` ⇒ cây. Kết luận `True`.

**Bài 3.** (1) `parent[x] = parent[parent[x]]`. (2) `x = parent[x]`. (3) Trả `x` khi `parent[x] == x`.

**Bài 4.** (1) `deque` chứa `(r, c, pr, pc)`. (2) Đánh dấu khi đẩy vào. (3) Hàng xóm cùng chữ, khác cha, đã thăm ⇒ `True`.

**Bài 5.** (1) Mỗi thành phần thuộc đúng một chữ. (2) Dùng `components_stats` hoặc DSU: cạnh đóng chu trình ⇒ thêm chữ của nó vào tập. (3) Không dừng sớm — duyệt hết mọi cạnh.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
a1 = (Solution().containsCycle(G("aa", "aa")), Solution().containsCycle(G("ab", "ba")))
print("Bài 1:", a1)
assert a1 == (True, False)

# Bài 2
stats = components_stats(G("aab", "aab", "bbb"))
print("Bài 2:", stats)
assert sorted(stats) == [("a", 4, 4), ("b", 5, 4)]
assert Solution().containsCycle(G("aab", "aab", "bbb")) is True

Bài 1: (True, False)
Bài 2: [('a', 4, 4), ('b', 5, 4)]


In [34]:
# Bài 3
def ex3_find_sol(parent, x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

par = [0, 0, 1, 2, 3, 5]
assert ex3_find_sol(par, 4) == 0 and ex3_find_sol(par, 5) == 5
assert par[4] in (0, 1, 2)          # đường đã được rút ngắn
print("Bài 3: OK, parent sau nén:", par)

Bài 3: OK, parent sau nén: [0, 0, 0, 2, 2, 5]


In [35]:
# Bài 4
def ex4_bfs_sol(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if seen[i][j]:
                continue
            seen[i][j] = True
            q = deque([(i, j, -1, -1)])
            while q:
                r, c, pr, pc = q.popleft()
                for nr, nc in neighbors(grid, r, c):
                    if (nr, nc) == (pr, pc):
                        continue
                    if seen[nr][nc]:
                        return True
                    seen[nr][nc] = True
                    q.append((nr, nc, r, c))
    return False

rng4 = random.Random(4)
for _ in range(600):
    m, n = rng4.randint(1, 6), rng4.randint(1, 6)
    g = [[rng4.choice("abc") for _ in range(n)] for _ in range(m)]
    assert ex4_bfs_sol(g) == count_v_e(g)
print("Bài 4: OK")

Bài 4: OK


In [36]:
# Bài 5
def ex5_cycle_letters_sol(grid):
    m, n = len(grid), len(grid[0])
    parent = list(range(m * n))
    out = set()
    for r in range(m):
        for c in range(n):
            for nr, nc in ((r, c + 1), (r + 1, c)):
                if nr < m and nc < n and grid[nr][nc] == grid[r][c]:
                    a, b = ex3_find_sol(parent, r * n + c), ex3_find_sol(parent, nr * n + nc)
                    if a == b:
                        out.add(grid[r][c])
                    else:
                        parent[a] = b
    return out

def ex5_check(grid):
    # đối chiếu: với mỗi chữ ch, thay các ô khác bằng token riêng biệt rồi hỏi containsCycle
    letters = {x for row in grid for x in row}
    res = set()
    for ch in letters:
        masked = [[x if x == ch else ("#", i, j) for j, x in enumerate(row)] for i, row in enumerate(grid)]
        if Solution().containsCycle(masked):
            res.add(ch)
    return res

rng5 = random.Random(5)
for _ in range(500):
    m, n = rng5.randint(1, 7), rng5.randint(1, 7)
    g = [[rng5.choice("abc") for _ in range(n)] for _ in range(m)]
    assert ex5_cycle_letters_sol(g) == ex5_check(g)
print("Bài 5: OK, ví dụ 1:", ex5_cycle_letters_sol(examples[0][0]), "| ví dụ 3:", ex5_cycle_letters_sol(examples[2][0]))

Bài 5: OK, ví dụ 1: {'a', 'b'} | ví dụ 3: set()


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def containsCycle(self, grid):
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                if seen[i][j]: continue
                ch = grid[i][j]; seen[i][j] = True
                stack = [(i, j, -1, -1)]
                while stack:
                    r, c, pr, pc = stack.pop()
                    for nr, nc in ((r+1, c), (r-1, c), (r, c+1), (r, c-1)):
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == ch and (nr, nc) != (pr, pc):
                            if seen[nr][nc]: return True
                            seen[nr][nc] = True
                            stack.append((nr, nc, r, c))
        return False
```

**Template "chu trình trong đồ thị vô hướng":** DFS/BFS nhớ cha (gặp đã thăm ≠ cha ⇒ chu trình) hoặc DSU (cạnh nối hai đỉnh cùng gốc ⇒ chu trình; mỗi cạnh xét một lần).

**Checklist trước khi submit:**
- [ ] Lưu và bỏ qua **ô cha**.
- [ ] Chỉ xét hàng xóm **cùng ký tự**.
- [ ] DSU: chỉ hướng phải + xuống.
- [ ] Không dùng đệ quy sâu với lưới 500×500.
- [ ] Thử nhanh: `["aa","aa"] → True`, `["aa"] → False`, ví dụ 3 → `False`.